In [1]:
import torch
import torch.nn as nn 
import torch.nn.functional as F

In [2]:
from transformers import BertTokenizer

model_name = 'bert-base-uncased'

# 해당 모델의 토크나이저(vocab / 규칙) 로드
tokenizer = BertTokenizer.from_pretrained(model_name)
tokenizer

BertTokenizer(name_or_path='bert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})

In [3]:
text = 'Here is the sentence I want embedding for'
tokenizer.encode(text)  # 텍스트 -> 토큰화 -> input_ids ([CLS] ... [SEP])

[101, 2182, 2003, 1996, 6251, 1045, 2215, 7861, 8270, 4667, 2005, 102]

In [4]:
tokenizer.tokenize(text)  # 워드피스 단위로 분리  # WordPiece 토큰 리스트로 분리

['here', 'is', 'the', 'sentence', 'i', 'want', 'em', '##bed', '##ding', 'for']

In [5]:
tokenizer(text)  # 문장을 토큰화 -> {input_ids: [...], token_type_ids: [...], attention_mask: [...]} 딕셔너리로 변환

{'input_ids': [101, 2182, 2003, 1996, 6251, 1045, 2215, 7861, 8270, 4667, 2005, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [6]:
from transformers import BertForMaskedLM  # 마스크된 모델을 추론

# Masked LM 헤드가 포함된 Bert 모델 : [MASK] 위치의 단어 예측하는 모델
model = BertForMaskedLM.from_pretrained(model_name)
model

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertForMaskedLM(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, el

In [7]:
# 토큰 처리 - 모델 입력 -> mask 토큰 추론
text = 'Score in a really fun [MASK].'

tokens = tokenizer.tokenize(text)  # WordPiece 토큰 리스트로 분해
print(tokens)

inputs = tokenizer(text, return_tensors='pt')  # 모델 입력용 텐서 생성 ({input_ids, token_type_ids, attention_mask})
inputs

['score', 'in', 'a', 'really', 'fun', '[MASK]', '.']


{'input_ids': tensor([[ 101, 3556, 1999, 1037, 2428, 4569,  103, 1012,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [8]:
# 특수토큰 3개 확인
print(tokenizer.cls_token_id)  # [CLS] 정수 ID
print(tokenizer.sep_token_id)  # [SEP] 정수 ID
print(tokenizer.mask_token_id)  # [MASK] 정수 ID

101
102
103


In [9]:
output = model(**inputs)  # forward 진행
output  # 출력 객체 확인 (주요 값: logits : (batch_size, seq_len, vocab_size))

MaskedLMOutput(loss=None, logits=tensor([[[ -6.5824,  -6.5396,  -6.5484,  ...,  -5.9696,  -5.7458,  -3.9639],
         [ -7.2907,  -7.3640,  -7.4438,  ...,  -7.0743,  -7.1102,  -8.3039],
         [-11.0806, -11.5142, -11.0337,  ..., -10.1931,  -9.1168,  -9.7695],
         ...,
         [ -6.8230,  -6.9543,  -6.8374,  ...,  -6.8570,  -5.0451,  -6.6299],
         [ -9.5275,  -9.3925,  -9.5250,  ...,  -8.5587,  -9.0284,  -2.9592],
         [-13.6764, -13.2981, -13.3741,  ...,  -9.9065, -10.0478, -11.1040]]],
       grad_fn=<ViewBackward0>), hidden_states=None, attentions=None)

In [10]:
from transformers import FillMaskPipeline  # [MASK] 채우기 전용 파이프라인 클래스

# 모델 + 토크나이저를 묶어 [MASK] 예측 파이프라인 객체 생성
pipe = FillMaskPipeline(model=model, tokenizer=tokenizer)

In [11]:
pipe(text)  # text문장의 [MASK]를 채울 후보 단어(top-k) 예측 결과 반환

[{'score': 0.8190984129905701,
  'token': 2126,
  'token_str': 'way',
  'sequence': 'score in a really fun way.'},
 {'score': 0.018992707133293152,
  'token': 2173,
  'token_str': 'place',
  'sequence': 'score in a really fun place.'},
 {'score': 0.008106553927063942,
  'token': 2051,
  'token_str': 'time',
  'sequence': 'score in a really fun time.'},
 {'score': 0.00716511020436883,
  'token': 2208,
  'token_str': 'game',
  'sequence': 'score in a really fun game.'},
 {'score': 0.0071624768897891045,
  'token': 2088,
  'token_str': 'world',
  'sequence': 'score in a really fun world.'}]

In [12]:
pipe('I want to [MASK] this morning.')

[{'score': 0.1426110714673996,
  'token': 5293,
  'token_str': 'forget',
  'sequence': 'i want to forget this morning.'},
 {'score': 0.1337175816297531,
  'token': 3342,
  'token_str': 'remember',
  'sequence': 'i want to remember this morning.'},
 {'score': 0.12416774034500122,
  'token': 3637,
  'token_str': 'sleep',
  'sequence': 'i want to sleep this morning.'},
 {'score': 0.05230763554573059,
  'token': 2707,
  'token_str': 'start',
  'sequence': 'i want to start this morning.'},
 {'score': 0.042421527206897736,
  'token': 2113,
  'token_str': 'know',
  'sequence': 'i want to know this morning.'}]

In [13]:
pipe('I wanna [MASK] this weekend.')

[{'score': 0.19230163097381592,
  'token': 2175,
  'token_str': 'go',
  'sequence': 'i wanna go this weekend.'},
 {'score': 0.0695551335811615,
  'token': 2147,
  'token_str': 'work',
  'sequence': 'i wanna work this weekend.'},
 {'score': 0.046094853430986404,
  'token': 8439,
  'token_str': 'celebrate',
  'sequence': 'i wanna celebrate this weekend.'},
 {'score': 0.043038126081228256,
  'token': 2994,
  'token_str': 'stay',
  'sequence': 'i wanna stay this weekend.'},
 {'score': 0.03529883176088333,
  'token': 2377,
  'token_str': 'play',
  'sequence': 'i wanna play this weekend.'}]

In [14]:
pipe('I wanna go to [MASK] this weekend.')

[{'score': 0.07085169106721878,
  'token': 2082,
  'token_str': 'school',
  'sequence': 'i wanna go to school this weekend.'},
 {'score': 0.05796119198203087,
  'token': 2147,
  'token_str': 'work',
  'sequence': 'i wanna go to work this weekend.'},
 {'score': 0.05154344066977501,
  'token': 2267,
  'token_str': 'college',
  'sequence': 'i wanna go to college this weekend.'},
 {'score': 0.034181222319602966,
  'token': 2277,
  'token_str': 'church',
  'sequence': 'i wanna go to church this weekend.'},
 {'score': 0.03271162882447243,
  'token': 7136,
  'token_str': 'vegas',
  'sequence': 'i wanna go to vegas this weekend.'}]

In [15]:
# 모델명에 맞는 토크나이저/모델 자동 선택
from transformers import AutoTokenizer, AutoModelForMaskedLM

model_name = 'bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)  # 모델에 맞는 토크나이저 자동 로드
model = AutoModelForMaskedLM.from_pretrained(model_name)  # 모델에 맞는 MaskedLM 모델 자동 로드

print(tokenizer.__class__.__name__)  # 실제 로드된 토크나이저 클래스 이름
print(model.__class__.__name__)  # 실제 로드된 모델 클래스 이름

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertTokenizer
BertForMaskedLM


In [16]:
# 모델명에 맞는 토크나이저/모델 자동 선택
from transformers import AutoTokenizer, AutoModelForNextSentencePrediction

model_name = 'bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)  # 모델에 맞는 토크나이저 자동 로드
model = AutoModelForNextSentencePrediction.from_pretrained(model_name)  # 모델에 맞는 NSP 모델 자동 로드


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] BertForNextSentencePrediction LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [17]:
sentence1 = "In Italy, pizza served in formal settings, such as at a restaurant, is presented unsliced."  # 문장 A
sentence2 = "pizza is eaten with the use of a knife and fork. In casual settings, however, it is cut into wedges to be eaten while held in the hand."  # 문장 B

# 두 문장을 한 쌍으로 인코딩 ([CLS] A [SEP] B [SEP])
inputs = tokenizer(sentence1, sentence2, return_tensors='pt')
inputs

{'input_ids': tensor([[  101,  1999,  3304,  1010, 10733,  2366,  1999,  5337, 10906,  1010,
          2107,  2004,  2012,  1037,  4825,  1010,  2003,  3591,  4895, 14540,
          6610,  2094,  1012,   102, 10733,  2003,  8828,  2007,  1996,  2224,
          1997,  1037,  5442,  1998,  9292,  1012,  1999, 10017, 10906,  1010,
          2174,  1010,  2009,  2003,  3013,  2046, 17632,  2015,  2000,  2022,
          8828,  2096,  2218,  1999,  1996,  2192,  1012,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [18]:
# NSP 입력 구성 확인 (tokens, input_ids, token_type_ids)
import pandas as pd

token_sent1 = tokenizer.tokenize(sentence1)  # 문장 1을 WordPiece 토큰으로 분해
token_sent2 = tokenizer.tokenize(sentence2)  # 문장 2을 WordPiece 토큰으로 분해

# NSP 입력 형태로 시퀀스 구성
tokens = ['[CLS]'] + token_sent1 + ['[SEP]'] + token_sent2 + ['[SEP]']
print(tokens)

pd.set_option('display.max_columns', None)  # DataFrame에서 column 생략 없이 전부 출력

pd.DataFrame([
    tokens,   # 사람이 읽을 수 있는 토큰 시퀀스
    inputs['input_ids'].squeeze(0).numpy(),  # (1, L) -> (L,) 줄이고, numpy 배열로 변환
    inputs['token_type_ids'].squeeze(0).numpy(),  # (1, L) -> (L,) 줄이고, numpy 배열로 변환
], index = ['tokens', 'input_ids', 'token_type_ids'])  # 행 이름 지정

['[CLS]', 'in', 'italy', ',', 'pizza', 'served', 'in', 'formal', 'settings', ',', 'such', 'as', 'at', 'a', 'restaurant', ',', 'is', 'presented', 'un', '##sl', '##ice', '##d', '.', '[SEP]', 'pizza', 'is', 'eaten', 'with', 'the', 'use', 'of', 'a', 'knife', 'and', 'fork', '.', 'in', 'casual', 'settings', ',', 'however', ',', 'it', 'is', 'cut', 'into', 'wedge', '##s', 'to', 'be', 'eaten', 'while', 'held', 'in', 'the', 'hand', '.', '[SEP]']


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57
tokens,[CLS],in,italy,",",pizza,served,in,formal,settings,",",such,as,at,a,restaurant,",",is,presented,un,##sl,##ice,##d,.,[SEP],pizza,is,eaten,with,the,use,of,a,knife,and,fork,.,in,casual,settings,",",however,",",it,is,cut,into,wedge,##s,to,be,eaten,while,held,in,the,hand,.,[SEP]
input_ids,101,1999,3304,1010,10733,2366,1999,5337,10906,1010,2107,2004,2012,1037,4825,1010,2003,3591,4895,14540,6610,2094,1012,102,10733,2003,8828,2007,1996,2224,1997,1037,5442,1998,9292,1012,1999,10017,10906,1010,2174,1010,2009,2003,3013,2046,17632,2015,2000,2022,8828,2096,2218,1999,1996,2192,1012,102
token_type_ids,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1


In [19]:
output = model(**inputs)  # forward 실행 -> logits
print(output)

prob = F.softmax(output[0], dim=1)  # 2클래스 logits -> 확률로 변환
print(prob)  # [NotNext, IsNext]

pred = torch.argmax(prob, dim=1).item()  # 확률이 더 큰 인덱스 선택 (0 또는 1)
print(pred)

NextSentencePredictorOutput(loss=None, logits=tensor([[ 6.3958, -6.3766]], grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)
tensor([[1.0000e+00, 2.8382e-06]], grad_fn=<SoftmaxBackward0>)
0


In [20]:
# sentence1과 전혀 상관없는 sentence3
sentence3 = 'The sky is blue due to the shorter wavelength of blue light.'

inputs = tokenizer(sentence1, sentence3, return_tensors='pt')  # (문장1, 문장3) 묶어 인코딩
output = model(**inputs)  # 순전파
prob = F.softmax(output[0], dim=1)  # logits -> 확률 ([NotNext, IsNext])
print(prob)  
pred = torch.argmax(prob, dim=1).item()  # 확률이 더 큰 인덱스 선택 (0 또는 1)
print(pred)

tensor([[1.2606e-04, 9.9987e-01]], grad_fn=<SoftmaxBackward0>)
1


In [22]:
# 문장 후보중 가장 이어질 문장 순위 매기기
candidates = [
    sentence2,
    sentence3, 
    'Pizza is one of the most popular food in the world.',
    'I enjoy playing football on weekends.'
]

scores = []

for s2 in candidates:
    # 두 문장을 한 쌍으로 인코딩([CLS] A [SEP] B [SEP])
    inp = tokenizer(sentence1, s2, return_tensors='pt')
    output = model(**inp)             # 순전파
    # logits(1, 2) -> 확률 ([[IsNext, NotNext]] 및 1차원(2,)) 변환
    prob = F.softmax(output.logits, dim=-1).squeeze(0)
    isnext = float(prob[0])  # IsNext(이어짐) 확률
    scores.append((isnext, s2))

scores.sort(reverse=True, key=lambda x: x[0])  # IsNext 확률 기준 내림차순 정렬
for p, s2 in scores:
    print(f"확률 : {p:.4f}, 문장 : {s2}")

확률 : 1.0000, 문장 : pizza is eaten with the use of a knife and fork. In casual settings, however, it is cut into wedges to be eaten while held in the hand.
확률 : 1.0000, 문장 : Pizza is one of the most popular food in the world.
확률 : 0.0001, 문장 : The sky is blue due to the shorter wavelength of blue light.
확률 : 0.0000, 문장 : I enjoy playing football on weekends.
